# Bridge Structural Health Monitoring — 3D VBI Simulation

This notebook runs the refactored 3D vehicle-bridge interaction simulation on Google Colab.
The code is imported from your Google Drive folder.

**Pipeline:**
1. Install OpenSeesPy + Python dependencies
2. Mount Google Drive and import the simulation package
3. Run 3D grillage + shell deck simulation with quarter-car VBI
4. Inject sensor noise & thermal effects
5. Export CSV dataset and validation plots

---
## Cell 1: Install OpenSeesPy and dependencies

In [ ]:
# Install system BLAS library needed by OpenSeesPy
!apt-get update -qq && apt-get install -y -qq libblas3 2>&1 | tail -5

# Install Python packages
!pip install -q openseespy numpy pandas scipy matplotlib tqdm scikit-learn joblib 2>&1 | tail -5

# Set LD_LIBRARY_PATH so OpenSeesPy can find libblas.so.3
import os
import site
blas_path = os.path.join(site.getsitepackages()[-1], 'openseespylinux', 'lib')
if os.path.isdir(blas_path):
    os.environ['LD_LIBRARY_PATH'] = f"{blas_path}:" + os.environ.get('LD_LIBRARY_PATH', '')
    print(f'LD_LIBRARY_PATH set to: {blas_path}')

print('Dependencies installed.')

---
## Cell 2: Mount Google Drive

In [ ]:
from google.colab import drive
import sys
import os

drive.mount('/content/drive')

# --- CONFIGURE THIS PATH ---
# Set this to the path of your bridge_SHM folder on Google Drive
REPO_PATH = '/content/drive/MyDrive/bridge_SHM'

if not os.path.exists(REPO_PATH):
    print(f'ERROR: Path {REPO_PATH} not found.')
    print('Please set REPO_PATH to the correct folder on your Drive.')
else:
    sys.path.insert(0, REPO_PATH)
    print(f'Added {REPO_PATH} to Python path.')
    print(f'Contents: {os.listdir(REPO_PATH)}')

---
## Cell 3: Import modules

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from digital_twin.config import (
    BRIDGE_CONFIG, SIM_CONFIG, VEHICLE_LIBRARY,
    VBI_CONFIG, ENV_CONFIG, TRAFFIC_CONFIG, DAMAGE_CASES,
)
from digital_twin.bridge_simulation import run_simulation
from digital_twin.utils import ensure_dir

print('All modules imported successfully.')

---
## Cell 4: Display configuration

In [ ]:
print('=== Bridge Configuration ===')
for k, v in BRIDGE_CONFIG.items():
    print(f'  {k}: {v}')

print(f'\n=== Damage Cases ===')
for name, cfg in DAMAGE_CASES.items():
    print(f'  {name}: {cfg}')

print(f'\n=== VBI Parameters ===')
print(f'  mode: {VBI_CONFIG.get("vbi_mode", "loose")}')
print(f'  iso_class: {VBI_CONFIG["iso_class"]}')
print(f'  max_iterations: {VBI_CONFIG["vbi_max_iterations"]}')
print(f'  tolerance: {VBI_CONFIG["vbi_tolerance"]}')
print(f'  Vehicle types: {list(VBI_CONFIG["vehicle_params"].keys())}')

print(f'\n=== Environmental ===')
print(f'  Thermal alpha: {ENV_CONFIG["thermal"]["alpha"]}')
print(f'  Noise accel RMS%: {ENV_CONFIG["noise"]["accel_rms_pct"]*100:.1f}%')

---
## Cell 5: Run simulation (1 damage case, 1 run)

This runs the 3D grillage + shell deck simulation with loose VBI coupling (fast ~1.2 min per run).
Each run covers 20 seconds of stochastic traffic on a 60m 3-span bridge.

⚠ Set `vbi_mode='tight'` in VBI_CONFIG for research-grade accuracy (slower: ~3.7 s/step).

In [ ]:
OUTPUT_DIR = '/content/drive/MyDrive/bridge_SHM_outputs'
ensure_dir(OUTPUT_DIR)

damage_name = 'healthy'
damage_case = DAMAGE_CASES[damage_name]

from copy import deepcopy
local_vbi = deepcopy(VBI_CONFIG)
local_vbi['vbi_mode'] = 'loose'  # ~1.2 min/run

print(f'Running: {damage_name} | run 1')
signal_df, traffic_df = run_simulation(
    bridge_config=BRIDGE_CONFIG,
    sim_config=SIM_CONFIG,
    vehicle_library=VEHICLE_LIBRARY,
    vbi_config=local_vbi,
    env_config=ENV_CONFIG,
    traffic_config=TRAFFIC_CONFIG,
    damage_case_name=damage_name,
    damage_case=damage_case,
    run_id=1,
)

signal_df['run_id'] = 1
signal_df['damage_case'] = damage_name

print(f'\nSignal rows: {len(signal_df)}')
print(f'Traffic rows: {len(traffic_df)}')
print(f'Columns: {list(signal_df.columns)}')
print(f'Sensor nodes: {sorted(signal_df.sensor_node.unique())}')
print(f'Completion ratio: {signal_df.completion_ratio.iloc[0]:.2f}')
print(f'Damage case: {signal_df.damage_case.iloc[0]}')
print(f'Label: {signal_df.label.iloc[0]}')

---
## Cell 6: Save CSV outputs to Drive

In [ ]:
signal_path = os.path.join(OUTPUT_DIR, 'bridge_response.csv')
traffic_path = os.path.join(OUTPUT_DIR, 'traffic_log.csv')

signal_df.to_csv(signal_path, index=False)
traffic_df.to_csv(traffic_path, index=False)

print(f'Saved signal data: {signal_path}')
print(f'Saved traffic log: {traffic_path}')

---
## Cell 7: Quick validation plots

Time series overlay of all 7 sensors for acceleration.

In [ ]:
fig, axes = plt.subplots(7, 1, figsize=(12, 14), sharex=True)
sensor_nodes = sorted(signal_df['sensor_node'].unique())

for ax, node in zip(axes, sensor_nodes):
    sub = signal_df[signal_df['sensor_node'] == node]
    ax.plot(sub['time'], sub['acceleration'], lw=0.5, color='tab:blue')
    ax.set_ylabel(f'Node {node}\nAccel (m/s²)')
    ax.grid(True, alpha=0.3)

axes[-1].set_xlabel('Time (s)')
fig.suptitle('Acceleration response — all 7 sensors', fontsize=14)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'validation_time_series.png'), dpi=150)
plt.show()
print('Saved: validation_time_series.png')

---
## Cell 8: FFT comparison (sensor 1)

In [ ]:
from scipy.fft import fft, fftfreq

sensor = sensor_nodes[0]
sub = signal_df[signal_df['sensor_node'] == sensor].copy()
acc = sub['acceleration'].values
dt_sim = SIM_CONFIG['dt']
n = len(acc)

yf = fft(acc)
xf = fftfreq(n, dt_sim)[:n // 2]

fig, axes = plt.subplots(2, 1, figsize=(10, 6))

axes[0].plot(sub['time'], acc, lw=0.5, color='tab:blue')
axes[0].set_ylabel('Acceleration (m/s²)')
axes[0].set_title(f'Time Domain — Sensor Node {sensor}')
axes[0].grid(True, alpha=0.3)

axes[1].plot(xf, 2.0 / n * np.abs(yf[:n // 2]), lw=0.8, color='tab:red')
axes[1].set_xlabel('Frequency (Hz)')
axes[1].set_ylabel('Amplitude')
axes[1].set_title('Frequency Domain (FFT)')
axes[1].set_xlim([0, 50])
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'validation_fft.png'), dpi=150)
plt.show()
print('Saved: validation_fft.png')

---
## Cell 9: Run full pipeline (all 4 damage cases × multiple runs)

Set `RUN_FULL = True` below to run all damage cases. This generates the
complete dataset for downstream ML training.

⚠ **Warning:** Each run takes ~1.2 min with loose coupling (default).
Set `RUNS_PER_CASE` lower (e.g. 2–3) for a quick test.
Set `vbi_mode='tight'` for research-grade accuracy (slower).


In [ ]:
RUN_FULL = False  # Set to True to run full pipeline
from copy import deepcopy
local_vbi = deepcopy(VBI_CONFIG)
local_vbi['vbi_mode'] = 'loose'  # fast; switch to 'tight' for research-grade

RUNS_PER_CASE = 3  # Runs per damage case

if RUN_FULL:
    all_signal = []
    all_traffic = []

    for damage_name, damage_case in DAMAGE_CASES.items():
        for run_id in range(RUNS_PER_CASE):
            print(f'\n--- {damage_name} | run {run_id + 1}/{RUNS_PER_CASE} ---')

            sig, traf = run_simulation(
                bridge_config=BRIDGE_CONFIG,
                sim_config=SIM_CONFIG,
                vehicle_library=VEHICLE_LIBRARY,
                vbi_config=local_vbi,
                env_config=ENV_CONFIG,
                traffic_config=TRAFFIC_CONFIG,
                damage_case_name=damage_name,
                damage_case=damage_case,
                run_id=run_id + 1,
            )

            sig['run_id'] = run_id + 1
            sig['damage_case'] = damage_name
            traf['run_id'] = run_id + 1
            traf['damage_case'] = damage_name

            all_signal.append(sig)
            all_traffic.append(traf)

    final_signal = pd.concat(all_signal, ignore_index=True)
    final_traffic = pd.concat(all_traffic, ignore_index=True)

    final_signal.to_csv(os.path.join(OUTPUT_DIR, 'full_bridge_response.csv'), index=False)
    final_traffic.to_csv(os.path.join(OUTPUT_DIR, 'full_traffic_log.csv'), index=False)

    print(f'\nFull pipeline complete.')
    print(f'Signal: {len(final_signal)} rows')
    print(f'Saved to: {OUTPUT_DIR}')
else:
    print('RUN_FULL is False. Set to True to execute full pipeline.')

---
## Cell 10: Cleanup — Free Colab memory

In [ ]:
import openseespy.opensees as ops
ops.wipe()
print('OpenSees model wiped.')